In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from matplotlib.ticker import MultipleLocator
from PIL import Image, ImageTk
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import collections

# Constants
HARTREE_TO_KCAL_MOL = 627.509474
TEMPERATURE_K_DEFAULT = 298.15
R_KCAL_MOL_K = 0.00198720425864083


def read_atom_labels(file_path):
    with open(file_path, 'r') as file:
        labels = [int(line.strip()) for line in file]
    return labels

def read_experimental_shifts(file_path):
    with open(file_path, 'r') as file:
        shifts = [float(line.strip()) for line in file]
    return shifts

def read_gaussian_file(file_path):
    with open(file_path, 'r') as file:
        data = file.readlines()
    return data

def extract_shielding_tensors(gaussian_data, labels):
    shielding_tensors = []
    for line in gaussian_data:
        if "Isotropic =" in line:
            parts = re.split(r'\s+', line.strip())
            try:
                atom_label = int(parts[0])
                if atom_label in labels:
                    shielding = float(parts[4])
                    shielding_tensors.append(shielding)
            except ValueError as e:
                print(f"Skipping line (ValueError): {line.strip()} due to error: {e}")
            except IndexError:
                print(f"Skipping line due to unexpected format (IndexError): {line.strip()}")
    return shielding_tensors

def calculate_average_shielding(shielding_tensors):
    if shielding_tensors:
        return np.mean(shielding_tensors)
    return None

def calculate_theoretical_shifts(average_shielding, conformation_shieldings):
    theoretical_shifts_calc = []
    if average_shielding is None:
        print("Warning: Average reference shielding is None. Cannot calculate theoretical shifts.")
        return [[] for _ in conformation_shieldings]
    for shielding_tensors_conf in conformation_shieldings:
        if shielding_tensors_conf:
            shifts = [average_shielding - shielding for shielding in shielding_tensors_conf]
            theoretical_shifts_calc.append(shifts)
        else:
            theoretical_shifts_calc.append([])
    return theoretical_shifts_calc

def calculate_deviations(theoretical_shifts_list, experimental_shifts_list):
    deviations_calc = []
    if not theoretical_shifts_list: return []
    for conf_theoretical_shifts in theoretical_shifts_list:
        if not conf_theoretical_shifts:
            deviations_calc.append([])
            continue
        current_exp_s = experimental_shifts_list[:len(conf_theoretical_shifts)]
        if len(current_exp_s) < len(conf_theoretical_shifts):
            print(f"Warning (Deviations): For a conformation, exp. shifts ({len(current_exp_s)}) < theo. shifts ({len(conf_theoretical_shifts)}).")
        deviation = [exp - calc for exp, calc in zip(current_exp_s, conf_theoretical_shifts)]
        deviations_calc.append(deviation)
    return deviations_calc

def extract_energies(gaussian_data):
    energies_found = []
    energy_pattern = re.compile(r'SCF Done:\s+E\([RU]?\w+\)\s*=\s*(-?\d+\.\d+)')
    for line in gaussian_data:
        match = energy_pattern.search(line)
        if match:
            energies_found.append(float(match.group(1)))
    return energies_found

def calculate_relative_energies():
    global relative_energies, energies
    if not energies or all(e is None for e in energies):
        messagebox.showinfo("Relative Energies", "Energies not extracted or all are invalid.")
        relative_energies = [None] * len(energies) if energies else []
        return
    valid_energies = [e for e in energies if e is not None]
    if not valid_energies:
        messagebox.showinfo("Relative Energies", "No valid (non-None) energies to calculate relative energies.")
        relative_energies = [None] * len(energies)
        return
    min_energy = min(valid_energies)
    relative_energies = [(energy - min_energy) * HARTREE_TO_KCAL_MOL if energy is not None else None for energy in energies]
    print(f"Relative Energies: {relative_energies}")

def show_relative_energies():
    if not relative_energies: messagebox.showinfo("Show Relative Energies", "Not calculated yet."); return
    show_content("Relative Energies", f"Relative Energies: {relative_energies}")

def show_content(title, content):
    new_window = tk.Toplevel(root)
    new_window.title(title)
    new_window.geometry("600x400") # Give pop-ups a decent default size
    text_frame = tk.Frame(new_window)
    text_frame.pack(expand=True, fill='both', padx=5, pady=5)
    text_widget = tk.Text(text_frame, wrap='none', height=15, width=70)
    text_widget.insert(tk.END, str(content))
    text_widget.config(state=tk.DISABLED)
    ys = tk.Scrollbar(text_frame, orient=tk.VERTICAL, command=text_widget.yview)
    text_widget['yscrollcommand'] = ys.set
    ys.pack(side=tk.RIGHT, fill=tk.Y)
    xs = tk.Scrollbar(text_frame, orient=tk.HORIZONTAL, command=text_widget.xview)
    text_widget['xscrollcommand'] = xs.set
    xs.pack(side=tk.BOTTOM, fill=tk.X)
    text_widget.pack(expand=True, fill='both')

def calculate_rmsd(deviations_list):
    rmsd_vals = []
    for deviation_conf in deviations_list:
        if not deviation_conf: rmsd_vals.append(np.nan); continue
        n = len(deviation_conf); ave = np.mean(deviation_conf)
        rmsd = np.sqrt(np.sum((np.array(deviation_conf) - ave) ** 2) / n) if n > 0 else np.nan
        rmsd_vals.append(rmsd)
    return rmsd_vals

def calculate_rmsd_button():
    global rmsd_values, deviations
    if deviations: rmsd_values = calculate_rmsd(deviations); print(f"RMSD Values: {rmsd_values}")
    else: messagebox.showinfo("Calculate RMSD", "Deviations not calculated yet.")

def show_rmsd():
    if not rmsd_values: messagebox.showinfo("Show RMSD", "Not calculated yet."); return
    show_content("RMSD Values", f"RMSD Values: {rmsd_values}")

def calculate_mae(theoretical_shifts_list, experimental_shifts_list):
    mae_vals = []
    if not theoretical_shifts_list: return []
    for conf_theoretical_shifts in theoretical_shifts_list:
        if not conf_theoretical_shifts: mae_vals.append(np.nan); continue
        current_exp_s = experimental_shifts_list[:len(conf_theoretical_shifts)]
        if len(current_exp_s) < len(conf_theoretical_shifts): mae_vals.append(np.nan); continue
        mae = np.mean([abs(shift - exp) for shift, exp in zip(conf_theoretical_shifts, current_exp_s)])
        mae_vals.append(mae)
    return mae_vals

def calculate_mae_button():
    global mae_values, theoretical_shifts, experimental_shifts
    if theoretical_shifts and experimental_shifts: mae_values = calculate_mae(theoretical_shifts, experimental_shifts); print(f"MAE Values: {mae_values}")
    else: messagebox.showinfo("Calculate MAE", "Theoretical or experimental shifts not available.")

def show_mae():
    if not mae_values: messagebox.showinfo("Show MAE", "Not calculated yet."); return
    show_content("MAE Values", f"MAE Values: {mae_values}")

def calculate_mad(deviations_list):
    mad_vals = []
    for deviation_conf in deviations_list:
        if not deviation_conf: mad_vals.append(np.nan); continue
        ave = np.mean(deviation_conf)
        mad = np.mean([abs(dev - ave) for dev in deviation_conf])
        mad_vals.append(mad)
    return mad_vals

def calculate_mad_button():
    global mad_values, deviations
    if deviations: mad_values = calculate_mad(deviations); print(f"MAD Values: {mad_values}")
    else: messagebox.showinfo("Calculate MAD", "Deviations not calculated yet.")

def show_mad():
    if not mad_values: messagebox.showinfo("Show MAD", "Not calculated yet."); return
    show_content("MAD Values", f"MAD Values: {mad_values}")

def calculate_rms(theoretical_shifts_list, experimental_shifts_list):
    rms_vals = []
    if not theoretical_shifts_list: return []
    for conf_theoretical_shifts in theoretical_shifts_list:
        if not conf_theoretical_shifts: rms_vals.append(np.nan); continue
        current_exp_s = experimental_shifts_list[:len(conf_theoretical_shifts)]
        if len(current_exp_s) < len(conf_theoretical_shifts): rms_vals.append(np.nan); continue
        rms = np.sqrt(np.mean([(shift - exp) ** 2 for shift, exp in zip(conf_theoretical_shifts, current_exp_s)]))
        rms_vals.append(rms)
    return rms_vals

def calculate_rms_button():
    global rms_values, theoretical_shifts, experimental_shifts
    if theoretical_shifts and experimental_shifts: rms_values = calculate_rms(theoretical_shifts, experimental_shifts); print(f"RMS Values: {rms_values}")
    else: messagebox.showinfo("Calculate RMS", "Theoretical or experimental shifts not available.")

def show_rms():
    if not rms_values: messagebox.showinfo("Show RMS", "Not calculated yet."); return
    show_content("RMS Values", f"RMS Values: {rms_values}")

def select_reference_labels():
    global reference_labels
    fp = filedialog.askopenfilename(title="Select Reference Atom Labels File");
    if fp:
        try: reference_labels = read_atom_labels(fp)
        except Exception as e: messagebox.showerror("File Error", f"Could not read reference labels: {e}")

def show_reference_labels():
    if not reference_labels: messagebox.showinfo("Show Labels", "Reference labels not loaded."); return
    show_content("Reference Labels (TMS)", f"Reference Labels (TMS): {reference_labels}")

def select_conformation_labels():
    global conformation_labels
    fp = filedialog.askopenfilename(title="Select Atom Labels for Conformations File")
    if fp:
        try: conformation_labels = read_atom_labels(fp)
        except Exception as e: messagebox.showerror("File Error", f"Could not read conformation labels: {e}")

def show_conformation_labels():
    if not conformation_labels: messagebox.showinfo("Show Labels", "Atom labels for conformations not loaded."); return
    show_content("Conformation Atom Labels", f"Conformation Atom Labels: {conformation_labels}")

def select_experimental_shifts():
    global experimental_shifts
    fp = filedialog.askopenfilename(title="Select Experimental Chemical Shifts File")
    if fp:
        try: experimental_shifts = read_experimental_shifts(fp)
        except Exception as e: messagebox.showerror("File Error", f"Could not read experimental shifts: {e}")

def show_experimental_shifts():
    if not experimental_shifts: messagebox.showinfo("Show Shifts", "Experimental shifts not loaded."); return
    show_content("Experimental Chemical Shifts", f"Experimental Chemical Shifts: {experimental_shifts}")

def select_gaussian_reference():
    global gaussian_reference_data
    fp = filedialog.askopenfilename(filetypes=[("Gaussian","*.log *.out *.txt")],title="Select Gaussian Reference File")
    if fp:
        try: gaussian_reference_data = read_gaussian_file(fp)
        except Exception as e: messagebox.showerror("File Error", f"Could not read Gaussian reference: {e}")

def show_gaussian_reference():
    if gaussian_reference_data: show_content("Gaussian Reference File", f"Loaded ({len(gaussian_reference_data)} lines).")
    else: messagebox.showinfo("Show Gaussian File", "No Gaussian Reference File loaded.")

def select_gaussian_conformations():
    global gaussian_conformations_data
    fps = filedialog.askopenfilenames(filetypes=[("Gaussian","*.log *.out *.txt")],title="Select Gaussian Conformation Files")
    if fps:
        temp_data = []
        try:
            for fp_conf in fps: temp_data.append(read_gaussian_file(fp_conf))
            gaussian_conformations_data = temp_data
        except Exception as e: messagebox.showerror("File Error", f"Error reading Gaussian conformation files: {e}")

def show_gaussian_conformations():
    if not gaussian_conformations_data: messagebox.showinfo("Show Gaussian Files", "No Conformation Files loaded."); return
    show_content("Gaussian Conformation Files", f"{len(gaussian_conformations_data)} files selected.")

def extract_reference_shielding():
    global extracted_reference_shielding, gaussian_reference_data, reference_labels
    if gaussian_reference_data and reference_labels:
        extracted_reference_shielding = extract_shielding_tensors(gaussian_reference_data, reference_labels)
        print(f"Extracted Ref Shieldings: {extracted_reference_shielding}")
        if not extracted_reference_shielding: messagebox.showwarning("Extraction", "No reference shieldings extracted.")
    else: messagebox.showinfo("Extract Shielding", "Gaussian ref data or labels missing.")

def show_reference_shielding():
    if not extracted_reference_shielding: messagebox.showinfo("Show Shielding", "Ref shieldings not extracted."); return
    show_content("Reference Shielding Tensors", f"Reference Shielding Tensors: {extracted_reference_shielding}")

def extract_conformation_shielding():
    global extracted_conformation_shielding, gaussian_conformations_data, conformation_labels
    if gaussian_conformations_data and conformation_labels:
        extracted_conformation_shielding = []
        all_empty = True
        for i, g_data_conf in enumerate(gaussian_conformations_data):
            shielding_tensors_conf = extract_shielding_tensors(g_data_conf, conformation_labels)
            if shielding_tensors_conf:
                all_empty = False
                if len(shielding_tensors_conf) != len(conformation_labels):
                    messagebox.showwarning("Extraction Mismatch",
                                           f"For Conformation {i+1}, #extracted shieldings ({len(shielding_tensors_conf)}) "
                                           f"!= #atom labels ({len(conformation_labels)}). Verify files.")
            extracted_conformation_shielding.append(shielding_tensors_conf)
        print(f"Extracted Conformation Shielding Tensors: {extracted_conformation_shielding}")
        if all_empty and gaussian_conformations_data: messagebox.showwarning("Extraction Warning", "No conf shieldings extracted.")
    else: messagebox.showinfo("Extract Shielding", "Gaussian conf data or labels not available.")

def show_conformation_shielding():
    if not extracted_conformation_shielding: messagebox.showinfo("Show Shielding", "Conf shieldings not extracted."); return
    content_str = "Conformation Shielding Tensors:\n"
    for i, s_tensors in enumerate(extracted_conformation_shielding):
        content_str += f"Conformation {i+1}: {s_tensors}\n"
    show_content("Conformation Shielding Tensors", content_str)

def calculate_average_reference_shielding():
    global average_reference_shielding, extracted_reference_shielding
    if extracted_reference_shielding:
        average_reference_shielding = calculate_average_shielding(extracted_reference_shielding)
        print(f"Avg Ref Shielding: {average_reference_shielding}")
        if average_reference_shielding is None: messagebox.showwarning("Calculation Warning", "Could not calculate avg ref shielding.")
    else: messagebox.showinfo("Calculate Average Shielding", "Ref shieldings not extracted yet.")

def show_average_reference_shielding():
    if average_reference_shielding is None: messagebox.showinfo("Show Avg. Shielding", "Avg ref shielding not calculated."); return
    show_content("Average Reference Shielding", f"Average Reference Shielding: {average_reference_shielding}")

def calculate_theoretical_shifts_button():
    global theoretical_shifts, average_reference_shielding, extracted_conformation_shielding
    if average_reference_shielding is not None and extracted_conformation_shielding:
        theoretical_shifts = calculate_theoretical_shifts(average_reference_shielding, extracted_conformation_shielding)
        print(f"Theoretical Chemical Shifts: {theoretical_shifts}")
        if not theoretical_shifts and extracted_conformation_shielding: messagebox.showwarning("Calc Warning", "Theo. shifts list empty.")
        elif extracted_conformation_shielding and all(not ts_conf for ts_conf in theoretical_shifts): messagebox.showwarning("Calc Warning", "Theo. shifts sub-lists all empty.")
    else: messagebox.showinfo("Calculate Theoretical Shifts", "Avg ref or conf shieldings missing.")

def show_theoretical_shifts():
    if not theoretical_shifts: messagebox.showinfo("Show Theo. Shifts", "Theo. shifts not calculated."); return
    content_str = "Theoretical Chemical Shifts:\n"
    for i, s_conf in enumerate(theoretical_shifts):
        content_str += f"Conformation {i+1}: {s_conf}\n"
    show_content("Theoretical Chemical Shifts", content_str)

def calculate_deviations_button():
    global deviations, theoretical_shifts, experimental_shifts
    if theoretical_shifts and experimental_shifts:
        deviations = calculate_deviations(theoretical_shifts, experimental_shifts)
        print(f"Deviations: {deviations}")
    else: messagebox.showinfo("Calculate Deviations", "Theo. or exp. shifts missing.")

def show_deviations():
    if not deviations: messagebox.showinfo("Show Deviations", "Deviations not calculated."); return
    content_str = "Deviations (Experimental - Theoretical):\n"
    for i, dev_conf in enumerate(deviations):
        content_str += f"Conformation {i+1}: {dev_conf}\n"
    show_content("Deviations", content_str)

def extract_energies_button():
    global energies, gaussian_conformations_data
    if gaussian_conformations_data:
        energies = []
        for g_data_conf in gaussian_conformations_data:
            energies_from_file = extract_energies(g_data_conf)
            energies.append(energies_from_file[-1] if energies_from_file else None)
        print(f"Energies: {energies}")
        if all(e is None for e in energies) and gaussian_conformations_data: messagebox.showwarning("Energy Extraction", "No SCF energies found.")
    else: messagebox.showinfo("Extract Energies", "Gaussian conf files not loaded.")

def show_energies():
    if not energies: messagebox.showinfo("Show Energies", "Energies not extracted."); return
    show_content("Energies (Hartree)", f"Energies: {energies}")

def _align_data_for_table(data_list, num_target, default_val=np.nan):
    if len(data_list) == num_target: return data_list
    aligned = list(data_list[:num_target]); aligned.extend([default_val]*(num_target-len(aligned)))
    return aligned

def generate_results_table(): # This function now only saves the CSV
    global relative_energies, rmsd_values, mae_values, mad_values, rms_values, theoretical_shifts
    num_confs = len(theoretical_shifts) if theoretical_shifts else len(relative_energies) if relative_energies else 0
    if not num_confs: messagebox.showerror("Error", "Insufficient data for results table."); return

    data = collections.OrderedDict()
    data["Conformation"] = [i + 1 for i in range(num_confs)] # Nomenclatura 1, 2, 3...
    data["Relative Energy (kcal/mol)"] = _align_data_for_table(relative_energies, num_confs)
    data["RMSD (ppm)"] = _align_data_for_table(rmsd_values, num_confs)
    data["MAD (ppm)"] = _align_data_for_table(mad_values, num_confs)
    data["RMS (ppm)"] = _align_data_for_table(rms_values, num_confs)
    data["MAE (ppm)"] = _align_data_for_table(mae_values, num_confs)

    try: df = pd.DataFrame(data)
    except ValueError as e: messagebox.showerror("DataFrame Error", f"Could not create results table: {e}"); return

    file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV","*.csv"),("TXT","*.txt")], title="Save Thermo & Spectro Data Table")
    if file_path:
        try:
            if file_path.endswith('.csv'): df.to_csv(file_path, index=False, na_rep='NaN', encoding='utf-8') # Added encoding
            else:
                with open(file_path, 'w', encoding='utf-8') as f: f.write(df.to_string(index=False, na_rep='NaN')) # Added encoding
            messagebox.showinfo("Save Successful", f"Table saved to: {file_path}")
        except Exception as e: messagebox.showerror("Save Error", f"Could not save table: {e}")

def generate_chemical_shift_table():
    global conformation_labels, experimental_shifts, theoretical_shifts, average_spectrum
    if not all([conformation_labels, experimental_shifts, theoretical_shifts, average_spectrum]):
        messagebox.showerror("Data Error", "Required data missing for chemical shift table."); return
    num_atoms = len(conformation_labels)
    if num_atoms == 0: messagebox.showerror("Data Error", "Atom labels count is zero."); return

    data = collections.OrderedDict()
    data["Atom Label"] = conformation_labels
    data["Experimental"] = _align_data_for_table(experimental_shifts, num_atoms)
    data["Boltzmann Weighted"] = _align_data_for_table(average_spectrum, num_atoms)
    for i, conf_theo_s in enumerate(theoretical_shifts):
        data[f"Theoretical {i+1}"] = _align_data_for_table(conf_theo_s, num_atoms) # Nomenclatura 1, 2, 3...

    try: df = pd.DataFrame(data)
    except ValueError as e: messagebox.showerror("DataFrame Error",f"Could not create chem shift table: {e}"); return
    print("\nChemical Shift Table (per Atom):\n", df.to_string(index=False, na_rep='N/A'))

    win = tk.Toplevel(root); win.title("Chemical Shift Table (per Atom)")
    win.geometry("800x500") # Larger popup for table
    tf_popup = tk.Frame(win); tf_popup.pack(expand=True,fill='both',padx=5,pady=5)
    txt_popup = tk.Text(tf_popup,wrap='none',height=20,width=100); txt_popup.insert(tk.END,df.to_string(index=False,na_rep='N/A')); txt_popup.config(state=tk.DISABLED)
    ys_popup = tk.Scrollbar(tf_popup,orient=tk.VERTICAL,command=txt_popup.yview); txt_popup['yscrollcommand']=ys_popup.set; ys_popup.pack(side=tk.RIGHT,fill=tk.Y)
    xs_popup = tk.Scrollbar(tf_popup,orient=tk.HORIZONTAL,command=txt_popup.xview); txt_popup['xscrollcommand']=xs_popup.set; xs_popup.pack(side=tk.BOTTOM,fill=tk.X)
    txt_popup.pack(expand=True,fill='both')

    fp_csv = filedialog.asksaveasfilename(defaultextension=".csv",filetypes=[("CSV","*.csv"),("TXT","*.txt")],title="Save Chemical Shift Table")
    if fp_csv:
        try:
            if fp_csv.endswith('.csv'): df.to_csv(fp_csv,index=False,na_rep='NaN', encoding='utf-8') # Added encoding
            else:
                with open(fp_csv,'w', encoding='utf-8') as f_out: f_out.write(df.to_string(index=False,na_rep='NaN')) # Added encoding
            messagebox.showinfo("Save Successful",f"Table saved to: {fp_csv}")
        except Exception as e: messagebox.showerror("Save Error",f"Could not save table: {e}")

def save_results_table(): pass # This function is defined but not directly used by buttons. generate_results_table handles saving.

def generate_theoretical_nmr_spectrum(lista_ts_param, atom_lbls_param):
    if not lista_ts_param: messagebox.showinfo("NMR Spectra","Theo. shifts not available."); return
    if not atom_lbls_param: messagebox.showinfo("NMR Spectra","Atom labels not available."); return
    directory = filedialog.askdirectory(title="Select Directory to Save Theoretical NMR Spectra")
    if not directory: return
    try: lista_ts_float = [[float(v) for v in sl] for sl in lista_ts_param]
    except Exception as e: messagebox.showerror("Data Error",f"Error converting theo. shifts: {e}"); return

    for i, ts_conf in enumerate(lista_ts_float):
        if not ts_conf: print(f"Skipping Conf. {i+1}: No theo. shifts data."); continue
        n_s = len(ts_conf); curr_lbls = [str(l) for l in atom_lbls_param[:n_s]]
        if len(curr_lbls) < n_s: curr_lbls.extend([f"Unlabeled_{k+1}" for k in range(len(curr_lbls),n_s)])
        try: paired = sorted(zip(ts_conf, curr_lbls), key=lambda x: x[0], reverse=True); vals=[p[0] for p in paired]; keys_s=[p[1] for p in paired]
        except Exception as e: print(f"Error Conf {i+1} spectrum data: {e}"); continue
        if not vals: print(f"Skipping Conf. {i+1}: No values to plot."); continue

        max_s,min_s=np.max(vals),np.min(vals); max_t,min_t=np.ceil(max_s),np.floor(min_s)
        if min_t>=max_t:min_t=max_t-1.0; max_t=min_t+1.0 if max_s==min_s else max_t # Ensure range
        int_tot=max_t-min_t; step=np.ceil(int_tot*0.1) if int_tot>0 else 0.1; step=max(step,0.1)
        try:
            tks=np.arange(max_t,min_t-step,-step);
            if not np.isclose(min_t,tks[-1]) and min_t<tks[-1]: tks=np.append(tks,min_t)
            tks=np.unique(np.round(tks,2)); tks=np.sort(tks)[::-1]
        except: tks=np.sort(np.linspace(min_t,max_t,num=5))[::-1]

        fig=plt.figure(figsize=(12,3)); plt.vlines(vals,0,1,color='red')
        plt.title(f'$^1$H-NMR Spectrum for Conformation {i+1}',fontsize=16,pad=20)
        plt.xlabel('Shifts (ppm)',fontsize=14)
        pad=(max_s-min_s)*0.05 or 0.5; plt.xlim(max_s+pad,min_s-pad)
        plt.xticks(ticks=tks,labels=[f"{t:.1f}" for t in tks],fontsize=12); ax=plt.gca()
        m_step=step/8.0 if step>0 else (max_t-min_t)/40.0 or 0.01; m_step=max(m_step,0.01)
        if m_step>0 and (max_t-min_t)>0 : ax.xaxis.set_minor_locator(MultipleLocator(m_step)); plt.minorticks_on()
        else: plt.minorticks_off()
        ax.tick_params(which='minor',length=3,width=0.8,labelsize=0)
        ax.yaxis.set_ticks([]); ax.yaxis.set_ticklabels([]); plt.ylim(0,1.5)
        plt.text((max_s+min_s)/2,1.35,'   '.join(keys_s),fontsize=12,color="black",ha='center',va='top')
        plt.tight_layout(pad=2)
        fp_fig=os.path.join(directory,f"Conformation_{i+1}_Theo_NMR_Spectrum.png")
        try: fig.savefig(fp_fig,bbox_inches='tight'); print(f"Saved Theo. Spectrum: {fp_fig}")
        except Exception as e: print(f"Error saving theo. spectrum Conf {i+1}: {e}")
        plt.close(fig)

def generate_experimental_nmr_spectrum(exp_s_param, atom_lbls_param):
    if not exp_s_param: messagebox.showinfo("Exp. NMR","No exp. shifts loaded."); return
    if not atom_lbls_param: messagebox.showinfo("Exp. NMR","Atom labels not available."); return
    directory = filedialog.askdirectory(title="Select Directory to Save Exp. NMR Spectrum")
    if not directory: return
    try: exp_s_float=[float(v) for v in exp_s_param]
    except Exception as e: messagebox.showerror("Data Error",f"Error converting exp. shifts: {e}"); return

    n_exp_s=len(exp_s_float)
    curr_lbls=[str(lbl) for lbl in atom_lbls_param[:n_exp_s]]
    if len(curr_lbls)<n_exp_s: curr_lbls.extend([f"Unlabeled_{k+1}" for k in range(len(curr_lbls),n_exp_s)])
    try:
        paired=sorted(zip(exp_s_float,curr_lbls),key=lambda x:x[0],reverse=True)
        vals=[p[0] for p in paired]; keys_s=[p[1] for p in paired]
    except Exception as e: print(f"Error exp. spectrum data: {e}"); return
    if not vals: messagebox.showinfo("Exp. NMR","No exp. shifts to plot."); return

    max_s,min_s=np.max(vals),np.min(vals); max_t,min_t=np.ceil(max_s),np.floor(min_s)
    if min_t>=max_t:min_t=max_t-1.0; max_t=min_t+1.0 if max_s==min_s else max_t
    int_tot=max_t-min_t; step=np.ceil(int_tot*0.1) if int_tot>0 else 0.1; step=max(step,0.1)
    try:
        tks=np.arange(max_t,min_t-step,-step)
        if not np.isclose(min_t,tks[-1]) and min_t<tks[-1]: tks=np.append(tks,min_t)
        tks=np.unique(np.round(tks,2)); tks=np.sort(tks)[::-1]
    except: tks=np.sort(np.linspace(min_t,max_t,num=5))[::-1]

    fig=plt.figure(figsize=(12,3)); plt.vlines(vals,0,1,color='blue')
    plt.title(r"Experimental $^1$H-NMR Spectrum",fontsize=16,pad=20)
    plt.xlabel("Shifts (ppm)",fontsize=14)
    pad=(max_s-min_s)*0.05 or 0.5; plt.xlim(max_s+pad,min_s-pad)
    plt.xticks(ticks=tks,labels=[f"{t:.1f}" for t in tks],fontsize=12); ax=plt.gca()
    m_step=step/8.0 if step>0 else (max_t-min_t)/40.0 or 0.01; m_step=max(m_step,0.01)
    if m_step>0 and (max_t-min_t)>0: ax.xaxis.set_minor_locator(MultipleLocator(m_step)); plt.minorticks_on()
    else: plt.minorticks_off()
    ax.tick_params(which='minor',length=3,width=0.8,labelsize=0)
    ax.yaxis.set_ticks([]); ax.yaxis.set_ticklabels([]); plt.ylim(0,1.5)
    plt.text((max_s+min_s)/2,1.35,'   '.join(keys_s),fontsize=12,color="black",ha='center',va='top')
    plt.tight_layout(pad=2)
    fp_fig_exp=os.path.join(directory,"Experimental_NMR_Spectrum.png")
    try: fig.savefig(fp_fig_exp,bbox_inches='tight'); print(f"Saved Exp. Spectrum: {fp_fig_exp}")
    except Exception as e: print(f"Error saving exp. spectrum: {e}")
    plt.close(fig)

def calculate_average_spectrum(theo_s_list, rel_e_list, temp=TEMPERATURE_K_DEFAULT):
    if not theo_s_list or not rel_e_list: messagebox.showwarning("Avg Spec Calc","Theo. shifts or rel. energies missing."); return []
    if len(theo_s_list)!=len(rel_e_list): messagebox.showwarning("Avg Spec Calc","Mismatch #theo. shifts & rel. energies."); return []

    valid_idx_e = [(i,E) for i,E in enumerate(rel_e_list) if E is not None]
    if not valid_idx_e: messagebox.showwarning("Avg Spec Calc","No valid rel. energies for weighting."); return []

    bf_raw=[]; RT = R_KCAL_MOL_K * temp
    for _,E_r in valid_idx_e:
        try: factor=np.exp(-E_r/RT); bf_raw.append(factor)
        except OverflowError: bf_raw.append(0.0)
    sum_bf=sum(bf_raw); actual_bw=[0.0]*len(theo_s_list)

    if sum_bf>1e-9:
        for i_raw,factor in enumerate(bf_raw): actual_bw[valid_idx_e[i_raw][0]]=factor/sum_bf
    elif valid_idx_e:
        eq_w=1.0/len(valid_idx_e)
        for vi,_ in valid_idx_e: actual_bw[vi]=eq_w
        messagebox.showwarning("Boltzmann W","Sum of factors near zero. Using equal weights for valid confs.")
    else: messagebox.showwarning("Boltzmann W","No valid energies; weights are zero."); return []

    n_nuc=0; first_s=next((s for s in theo_s_list if s),None)
    if first_s: n_nuc=len(first_s)
    else: messagebox.showwarning("Avg Spec Calc","No valid theo. shift data."); return []

    avg_s_calc=[np.nan]*n_nuc # Initialize with NaN
    for nuc_i in range(n_nuc):
        w_sum_n,sum_w_n = 0.0,0.0
        for struct_i,conf_s in enumerate(theo_s_list):
            if actual_bw[struct_i]>0 and conf_s and nuc_i<len(conf_s):
                s_val=conf_s[nuc_i]
                if s_val is not None and not np.isnan(s_val):
                    w_sum_n+=s_val*actual_bw[struct_i]; sum_w_n+=actual_bw[struct_i]
        if sum_w_n>1e-9: avg_s_calc[nuc_i]=w_sum_n/sum_w_n
    print(f"Calc Boltzmann W: {actual_bw}"); print(f"Calc Avg Spec: {avg_s_calc}")
    return avg_s_calc

def calculate_average_spectrum_button():
    global average_spectrum, theoretical_shifts, relative_energies
    if theoretical_shifts and relative_energies:
        if any(e is not None for e in relative_energies):
            average_spectrum = calculate_average_spectrum(theoretical_shifts, relative_energies)
            if average_spectrum: messagebox.showinfo("Avg Spectrum","Calculated successfully.")
            else: messagebox.showwarning("Avg Spectrum","Calculation failed or empty data.")
        else: messagebox.showwarning("Avg Spectrum","Rel. energies not properly calculated."); average_spectrum=[]
    else: messagebox.showinfo("Avg Spectrum","Theo. shifts or rel. energies not available."); average_spectrum=[]

def show_average_spectrum():
    if not average_spectrum: messagebox.showinfo("Show Avg Spectrum","Not calculated or empty."); return
    show_content("Boltzmann Weighted Avg Spectrum",f"Avg Spectrum: {average_spectrum}")

def calculate_average_spectrum_deviations(avg_s, exp_s):
    if not avg_s or not exp_s: return []
    aligned_exp=list(exp_s[:len(avg_s)]);
    if len(aligned_exp)<len(avg_s): aligned_exp.extend([np.nan]*(len(avg_s)-len(aligned_exp)))
    dev_avg=[]
    for e,a in zip(aligned_exp,avg_s):
        if e is not None and not np.isnan(e) and a is not None and not np.isnan(a): dev_avg.append(e-a)
        else: dev_avg.append(np.nan)
    return dev_avg

def calculate_average_spectrum_statistics(avg_s, exp_s):
    if not avg_s or not exp_s: messagebox.showwarning("Avg Spec Stats","Avg spectrum or exp. shifts missing."); return (np.nan,)*4
    dev_stats=calculate_average_spectrum_deviations(avg_s,exp_s)
    v_dev=[d for d in dev_stats if d is not None and not np.isnan(d)]
    if not v_dev: messagebox.showwarning("Avg Spec Stats","No valid deviations for stats."); return (np.nan,)*4
    n,ave_d=len(v_dev),np.mean(v_dev)
    rmsd=np.sqrt(np.sum((np.array(v_dev)-ave_d)**2)/n) if n>0 else np.nan
    mae=np.mean([abs(d) for d in v_dev]) if n>0 else np.nan
    mad=np.mean([abs(d-ave_d) for d in v_dev]) if n>0 else np.nan
    rms=np.sqrt(np.mean([d**2 for d in v_dev])) if n>0 else np.nan
    return rmsd,mae,mad,rms

def calculate_average_spectrum_statistics_button():
    global average_spectrum,experimental_shifts,avg_spec_stats_results; avg_spec_stats_results=None
    if average_spectrum and experimental_shifts:
        rmsd,mae,mad,rms = calculate_average_spectrum_statistics(average_spectrum,experimental_shifts)
        if not all(np.isnan([rmsd,mae,mad,rms])):
            avg_spec_stats_results=(rmsd,mae,mad,rms)
            print(f"\nAvg Spectrum Stats (vs Exp):\n  RMSD: {rmsd:.3f} ppm\n  MAE: {mae:.3f} ppm\n  MAD: {mad:.3f} ppm\n  RMS: {rms:.3f} ppm")
            messagebox.showinfo("Avg Spectrum Stats","Calculated. See console or 'Show'.")
        else: messagebox.showwarning("Avg Spectrum Stats","Could not calculate stats (all NaN).")
    else: messagebox.showinfo("Avg Spectrum Stats","Avg spectrum or exp. shifts not available.")

def show_average_spectrum_statistics():
    global avg_spec_stats_results
    if avg_spec_stats_results:
        rmsd, mae, mad, rms = avg_spec_stats_results
        stats_text = (f"Stats for Boltzmann Weighted Avg Spectrum (vs Exp):\n\n"
                      f"  RMSD (of deviations): {rmsd:.3f} ppm\n"
                      f"  MAE (|exp-avg|): {mae:.3f} ppm\n"
                      f"  MAD (of deviations): {mad:.3f} ppm\n"
                      f"  RMS (sqrt(mean((exp-avg)^2))): {rms:.3f} ppm")
        show_content("Average Spectrum Statistics", stats_text)
    else: messagebox.showinfo("Show Avg. Spectrum Stats", "Stats not calculated or results invalid.")

# --- Funções de placeholder (substitutas) ---
# Substitua o conteúdo destas funções pela sua implementação real.
def plot_relative_energies():
    """
    TODO: Implementar a lógica para mostrar o gráfico de energias relativas.
    Por exemplo, usando matplotlib para criar um gráfico.
    """
    global relative_energies
    if not relative_energies or all(e is None for e in relative_energies):
        messagebox.showinfo("Plot Relative Energies", "Relative energies not calculated or available to plot.")
        return
    
    # Exemplo básico de plotagem (requer que o usuário tenha matplotlib instalado e configurado para Tkinter se for embutido)
    # Para um pop-up simples com o gráfico:
    try:
        fig, ax = plt.subplots()
        confs = range(1, len(relative_energies) + 1)
        valid_energies = [e if e is not None else np.nan for e in relative_energies] # Substitui None por NaN para plotar
        
        ax.plot(confs, valid_energies, marker='o', linestyle='-')
        ax.set_xlabel("Conformation")
        ax.set_ylabel("Relative Energy (kcal/mol)")
        ax.set_title("Relative Energies of Conformations")
        ax.xaxis.set_major_locator(MultipleLocator(1)) # Garante ticks inteiros para conformações
        plt.grid(True)
        plt.show() # Mostra o gráfico em uma nova janela do matplotlib
    except Exception as e:
        messagebox.showerror("Plot Error", f"Could not plot relative energies: {e}")
    # messagebox.showinfo("Plot Relative Energies", "Functionality to plot relative energies needs to be implemented here.")

def save_plot():
    """
    TODO: Implementar a lógica para salvar um gráfico.
    Você precisará decidir qual gráfico salvar e como.
    """
    # Exemplo: Se você quiser salvar o gráfico de energias relativas que `plot_relative_energies` poderia gerar:
    global relative_energies
    if not relative_energies or all(e is None for e in relative_energies):
        messagebox.showinfo("Save Plot", "No data available to generate and save plot.")
        return

    file_path = filedialog.asksaveasfilename(
        defaultextension=".png",
        filetypes=[("PNG files", "*.png"), ("JPEG files", "*.jpg"), ("All files", "*.*")],
        title="Save Relative Energies Plot"
    )
    if not file_path:
        return

    try:
        fig, ax = plt.subplots()
        confs = range(1, len(relative_energies) + 1)
        valid_energies = [e if e is not None else np.nan for e in relative_energies]
        
        ax.plot(confs, valid_energies, marker='o', linestyle='-')
        ax.set_xlabel("Conformation")
        ax.set_ylabel("Relative Energy (kcal/mol)")
        ax.set_title("Relative Energies of Conformations")
        ax.xaxis.set_major_locator(MultipleLocator(1))
        plt.grid(True)
        plt.savefig(file_path)
        plt.close(fig) # Fecha a figura para liberar memória
        messagebox.showinfo("Save Plot", f"Plot saved to: {file_path}")
    except Exception as e:
        messagebox.showerror("Save Plot Error", f"Could not save plot: {e}")
    # messagebox.showinfo("Save Plot", "Functionality to save plot needs to be implemented here.")


# Global variables
reference_labels,conformation_labels,experimental_shifts = [],[],[]
gaussian_reference_data,gaussian_conformations_data = None,[]
extracted_reference_shielding,extracted_conformation_shielding = [],[]
average_reference_shielding = None
theoretical_shifts,deviations,energies,relative_energies = [],[],[],[]
rmsd_values,mae_values,mad_values,rms_values = [],[],[],[]
average_spectrum,avg_spec_stats_results = [],None


# Start of the GUI
root = tk.Tk()
root.title("CACSTD")
root.geometry("1000x700") # Set a larger initial size

try:
    # Certifique-se de que o arquivo "Logo.png" está no mesmo diretório do script
    # ou forneça o caminho completo para o arquivo.
    logo = Image.open("Logo.png")
    logo_imgtk = ImageTk.PhotoImage(logo)
    root.iconphoto(False, logo_imgtk)
except Exception as e_logo: print(f"Could not load logo: {e_logo}")

# Main frame setup for scrolling content
main_canvas_frame = tk.Frame(root)
main_canvas_frame.pack(fill="both", expand=True)
canvas = tk.Canvas(main_canvas_frame)
canvas.pack(side="left", fill="both", expand=True)
scrollbar = tk.Scrollbar(main_canvas_frame, orient="vertical", command=canvas.yview)
scrollbar.pack(side="right", fill="y")
canvas.configure(yscrollcommand=scrollbar.set)
content_frame_for_canvas = tk.Frame(canvas)
content_frame_for_canvas.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
canvas.create_window((0,0), window=content_frame_for_canvas, anchor="nw")

# Overall container for sections
sections_container_frame = tk.Frame(content_frame_for_canvas)
sections_container_frame.pack(pady=10, padx=10, fill="x", expand=True)

# --- Helper functions for GUI layout ---
NUM_BUTTON_COLUMNS = 2 # Define number of columns for button pairs

def create_button_pair_frame(parent_container, text, command, show_command):
    """Creates a frame containing a main action button and an optional Show button."""
    pair_frame = tk.Frame(parent_container)
    # No pack or grid here, will be done by the caller (add_button_to_grid)

    main_button = tk.Button(pair_frame, text=text, command=command, anchor='w', justify='left')
    main_button.pack(side='left', expand=True, fill='x', padx=(0, 2 if show_command else 0))

    if show_command:
        show_button = tk.Button(pair_frame, text="Show", command=show_command, width=8) # Standardized width for "Show"
        show_button.pack(side='left')
    return pair_frame

def place_buttons_in_grid(section_lf, button_definitions):
    """
    Places buttons (either pairs or single full-width) into a grid within the LabelFrame.
    button_definitions is a list of tuples: (text, command, show_command_or_None, type)
    type can be 'pair' or 'single'.
    """
    grid_container = tk.Frame(section_lf)
    grid_container.pack(fill='x', expand=True, padx=5, pady=5)
    for i in range(NUM_BUTTON_COLUMNS):
        grid_container.columnconfigure(i, weight=1)

    row_idx = 0
    col_idx = 0
    for item_def in button_definitions:
        text, command, show_command, item_type = item_def

        if item_type == 'pair':
            btn_frame = create_button_pair_frame(grid_container, text, command, show_command)
            btn_frame.grid(row=row_idx, column=col_idx, sticky='ew', padx=2, pady=2)
            col_idx += 1
            if col_idx >= NUM_BUTTON_COLUMNS:
                col_idx = 0
                row_idx += 1
        elif item_type == 'single':
            # If there's something in the current row, move to next row for full-width button
            if col_idx != 0:
                col_idx = 0
                row_idx += 1
            btn = tk.Button(grid_container, text=text, command=command, anchor='w', justify='left')
            btn.grid(row=row_idx, column=0, columnspan=NUM_BUTTON_COLUMNS, sticky='ew', padx=2, pady=2)
            row_idx += 1 # Full-width button takes the whole row
            col_idx = 0 # Reset for next item

# --- Inputs Section ---
inputs_lf = tk.LabelFrame(sections_container_frame, text="Inputs", padx=10, pady=10)
inputs_lf.pack(pady=5, fill='x', expand=True)
input_buttons_defs = [
    ("Read Reference Labels (TMS)", select_reference_labels, show_reference_labels, 'pair'),
    ("Read Atom Labels (Conformations)", select_conformation_labels, show_conformation_labels, 'pair'),
    ("Read Experimental Chemical Shifts", select_experimental_shifts, show_experimental_shifts, 'pair'),
    ("Read Gaussian File (Reference)", select_gaussian_reference, show_gaussian_reference, 'pair'),
    ("Read Gaussian Files (Conformations)", select_gaussian_conformations, show_gaussian_conformations, 'pair'),
]
place_buttons_in_grid(inputs_lf, input_buttons_defs)

# --- Thermodynamic Analysis Section ---
thermo_lf = tk.LabelFrame(sections_container_frame, text="Thermodynamic Analysis", padx=10, pady=10)
thermo_lf.pack(pady=5, fill='x', expand=True)
thermo_buttons_defs = [
    ("Extract Energies", extract_energies_button, show_energies, 'pair'),
    ("Calculate Relative Energies", calculate_relative_energies, show_relative_energies, 'pair'),
    ("Show Relative Energies Plot", plot_relative_energies, None, 'single'), # CORRIGIDO
    ("Save Relative Energies Plot", save_plot, None, 'single'),             # CORRIGIDO
]
place_buttons_in_grid(thermo_lf, thermo_buttons_defs) # Esta chamada agora usa as definições corrigidas


# --- Spectroscopic Analysis (per Conformation) Section ---
spectro_lf = tk.LabelFrame(sections_container_frame, text="Spectroscopic Analysis (per Conformation)", padx=10, pady=10)
spectro_lf.pack(pady=5, fill='x', expand=True)
spectro_buttons_defs = [
    ("Extract Reference Shielding Tensors", extract_reference_shielding, show_reference_shielding, 'pair'),
    ("Extract Conformation Shielding Tensors", extract_conformation_shielding, show_conformation_shielding, 'pair'),
    ("Calculate Average Shielding (Reference)", calculate_average_reference_shielding, show_average_reference_shielding, 'pair'),
    ("Calculate Theoretical Chemical Shifts", calculate_theoretical_shifts_button, show_theoretical_shifts, 'pair'),
    ("Calculate Deviations (Exp - Theo)", calculate_deviations_button, show_deviations, 'pair'),
    ("Calculate RMSD (of deviations)", calculate_rmsd_button, show_rmsd, 'pair'),
    ("Calculate MAE (Exp vs Theo)", calculate_mae_button, show_mae, 'pair'),
    ("Calculate MAD (of deviations)", calculate_mad_button, show_mad, 'pair'),
    ("Calculate RMS (Exp vs Theo)", calculate_rms_button, show_rms, 'pair'), # Last pair for this row if NUM_COLS=2
    ("Generate Theoretical NMR Spectra (per Conf)", lambda: generate_theoretical_nmr_spectrum(theoretical_shifts, conformation_labels), None, 'single'),
    ("Generate Experimental NMR Spectrum", lambda: generate_experimental_nmr_spectrum(experimental_shifts, conformation_labels), None, 'single'),
]
place_buttons_in_grid(spectro_lf, spectro_buttons_defs)

# --- Boltzmann Weighted Average Spectrum Analysis Section ---
boltzmann_lf = tk.LabelFrame(sections_container_frame, text="Boltzmann Weighted Average Spectrum Analysis", padx=10, pady=10)
boltzmann_lf.pack(pady=5, fill='x', expand=True)
boltzmann_buttons_defs = [
    ("Calculate Boltzmann Avg. Spectrum", calculate_average_spectrum_button, show_average_spectrum, 'pair'),
    ("Calculate Statistics for Avg. Spectrum", calculate_average_spectrum_statistics_button, show_average_spectrum_statistics, 'pair'),
]
place_buttons_in_grid(boltzmann_lf, boltzmann_buttons_defs)

# --- Result Tables & Export Section ---
results_lf = tk.LabelFrame(sections_container_frame, text="Result Tables & Export", padx=10, pady=10)
results_lf.pack(pady=5, fill='x', expand=True)
results_buttons_defs = [
    ("Save Thermo & Spectro Data Table (per Conf)", generate_results_table, None, 'single'),
    ("Show & Save Chemical Shift Table (per Atom)", generate_chemical_shift_table, None, 'single'),
]
place_buttons_in_grid(results_lf, results_buttons_defs)

root.mainloop()

In [ ]:
pip install jupyter



In [ ]:
pip install jupyter